In [ ]:
import numpy as np
from math import erf, sqrt

# =====================================================
# 3. Asian Call Option with Control Variate Method
# =====================================================
# 本題目標：
# 使用 Monte Carlo 定價 arithmetic average Asian call，
# 並用 geometric average Asian call 作為 control variate。
#
# arithmetic average Asian call 沒有簡單 closed-form solution，
# 但 geometric average Asian call 有解析解，
# 且兩者 payoff 高度相關，因此可用來降低 variance。


# -----------------------------------------------------
# Standard Normal CDF (不用 scipy)
# -----------------------------------------------------
def normal_cdf(x):
    return 0.5 * (1 + erf(x / sqrt(2)))


# -----------------------------------------------------
# Geometric Asian Call Closed-Form Price
# -----------------------------------------------------
# 使用講義中的 geometric average Asian call formula：
#
# C_ga = S0 exp(-(r/2 + sigma^2/12)T) N(d1)
#        - K exp(-rT) N(d2)
#
def geometric_asian_call_price(S0, K, r, sigma, T):

    sigma_g = sigma * np.sqrt(T / 3)

    d1 = (
        np.log(S0 / K)
        + 0.5 * (r + sigma**2 / 6) * T
    ) / sigma_g

    d2 = d1 - sigma_g

    price = (
        S0 * np.exp(-(r / 2 + sigma**2 / 12) * T)
        * normal_cdf(d1)
        - K * np.exp(-r * T)
        * normal_cdf(d2)
    )

    return price


# -----------------------------------------------------
# Asian Call with Control Variate
# -----------------------------------------------------
def price_asian_call_control_variate(
    S0=10,
    K=10,
    r=0.04,
    sigma=0.40,
    T=1/12,
    n_paths=100000,
    n_steps=252,
    seed=42
):

    np.random.seed(seed)

    dt = T / n_steps

    arithmetic_payoffs = []
    geometric_payoffs = []

    # =================================================
    # Monte Carlo Simulation
    # =================================================
    for _ in range(n_paths):

        S = S0
        path_prices = []

        # 在 risk-neutral measure 下模擬股價路徑
        for _ in range(n_steps):

            z = np.random.randn()

            S = S * np.exp(
                (r - 0.5 * sigma**2) * dt
                + sigma * np.sqrt(dt) * z
            )

            path_prices.append(S)

        path_prices = np.array(path_prices)

        # ---------------------------------------------
        # Arithmetic Average
        # ---------------------------------------------
        arithmetic_avg = np.mean(path_prices)

        # Arithmetic Asian Call payoff
        arithmetic_payoff = np.exp(-r * T) * max(
            arithmetic_avg - K,
            0
        )

        # ---------------------------------------------
        # Geometric Average
        # ---------------------------------------------
        geometric_avg = np.exp(
            np.mean(np.log(path_prices))
        )

        # Geometric Asian Call payoff
        geometric_payoff = np.exp(-r * T) * max(
            geometric_avg - K,
            0
        )

        arithmetic_payoffs.append(arithmetic_payoff)
        geometric_payoffs.append(geometric_payoff)

    arithmetic_payoffs = np.array(arithmetic_payoffs)
    geometric_payoffs = np.array(geometric_payoffs)

    # =================================================
    # Ordinary Monte Carlo Estimator
    # =================================================
    ordinary_price = np.mean(arithmetic_payoffs)

    ordinary_sd = np.std(
        arithmetic_payoffs,
        ddof=1
    )

    ordinary_se = ordinary_sd / np.sqrt(n_paths)

    ordinary_ci_95 = (
        ordinary_price - 1.96 * ordinary_se,
        ordinary_price + 1.96 * ordinary_se
    )

    # =================================================
    # Geometric Asian Call Closed-Form Price
    # =================================================
    geometric_price = geometric_asian_call_price(
        S0, K, r, sigma, T
    )

    # =================================================
    # Estimate Optimal Theta*
    # =================================================
    # theta* = Cov(X,Y) / Var(Y)
    #
    # X = arithmetic Asian payoff
    # Y = geometric Asian payoff
    #
    cov_matrix = np.cov(
        arithmetic_payoffs,
        geometric_payoffs,
        ddof=1
    )

    cov_xy = cov_matrix[0, 1]
    var_y = cov_matrix[1, 1]

    theta_star = cov_xy / var_y

    # =================================================
    # Control Variate Estimator
    # =================================================
    # X_cv = X + theta*(E[Y] - Y)
    #
    cv_payoffs = arithmetic_payoffs + theta_star * (
        geometric_price - geometric_payoffs
    )

    cv_price = np.mean(cv_payoffs)

    cv_sd = np.std(
        cv_payoffs,
        ddof=1
    )

    cv_se = cv_sd / np.sqrt(n_paths)

    cv_ci_95 = (
        cv_price - 1.96 * cv_se,
        cv_price + 1.96 * cv_se
    )

    # =================================================
    # Return Results
    # =================================================
    return {

        # Ordinary Monte Carlo
        "ordinary_MC_price": ordinary_price,
        "ordinary_MC_sd": ordinary_sd,
        "ordinary_MC_se": ordinary_se,
        "ordinary_MC_95% CI": ordinary_ci_95,

        # Geometric Asian Call
        "geometric_asian_price": geometric_price,

        # Optimal theta*
        "theta_star": theta_star,

        # Control Variate Results
        "control_variate_price": cv_price,
        "control_variate_sd": cv_sd,
        "control_variate_se": cv_se,
        "control_variate_95% CI": cv_ci_95
    }


# =====================================================
# Run
# =====================================================

result = price_asian_call_control_variate()

print("===== 3. Asian Call with Control Variate =====")

for key, value in result.items():
    print(key, ":", value)